In [ ]:
import os
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
import segmentation_models_pytorch as smp
from torchmetrics.classification import BinaryConfusionMatrix

class Config:
    def __init__(
        self, 
        active_model="unet", 
        encoder_name="resnet34", 
        loss_name="bce",
        learning_rate=1e-4, 
        train_batch_size=8, 
        eval_batch_size=8,
        image_dir="./data/train/images",
        mask_dir="./data/train/masks",
        results_dir="./results",
        graphs_dir="./graphs",
        k_folds=5,
        num_classes=1,
        encoder_weights="imagenet",
        included_filth_types=None,
        include_clean_data=True,
        epochs=50,
        patience=5
    ):
        self.IMAGE_DIR = image_dir
        self.MASK_DIR = mask_dir
        self.RESULTS_DIR = results_dir
        self.GRAPHS_DIR = graphs_dir
        
        self.K_FOLDS = k_folds
        self.NUM_CLASSES = num_classes 
        
        self.ACTIVE_MODEL = active_model.lower() 
        self.ENCODER = encoder_name.lower()      
        self.ENCODER_WEIGHTS = encoder_weights
        self.LOSS_NAME = loss_name.lower()       
        
        self.INCLUDED_FILTH_TYPES = included_filth_types if included_filth_types is not None else ["meat", "vegetables"]
        self.INCLUDE_CLEAN_DATA = include_clean_data 
        
        self.LEARNING_RATE = learning_rate
        self.TRAIN_BATCH_SIZE = train_batch_size
        self.EVAL_BATCH_SIZE = eval_batch_size
        self.EPOCHS = epochs
        self.PATIENCE = patience

    def get_experiment_name(self):
        return f"{self.ACTIVE_MODEL}_{self.ENCODER}_{self.LOSS_NAME}"

class UnifiedSegmentationDataset(Dataset):
    def __init__(self, filenames, img_dir, mask_dir, preprocessing_fn=None):
        self.filenames = filenames
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.preprocessing_fn = preprocessing_fn

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        image_path = os.path.join(self.img_dir, fname)
        mask_path = os.path.join(self.mask_dir, os.path.splitext(fname)[0] + ".png")

        image_pil = Image.open(image_path).convert("RGB").resize((512, 512), Image.BILINEAR)
        mask_pil = Image.open(mask_path).convert("RGB").resize((512, 512), Image.NEAREST)
        
        image = np.array(image_pil)
        mask_rgb = np.array(mask_pil)
        
        mask_binarized = np.any(mask_rgb > 0, axis=-1).astype(np.float32)

        if self.preprocessing_fn:
            image = self.preprocessing_fn(image)
            if image.shape[-1] == 3:
                image = image.transpose(2, 0, 1)

        image_tensor = torch.tensor(image, dtype=torch.float32)
        mask_tensor = torch.tensor(mask_binarized, dtype=torch.float32).unsqueeze(0) 

        return image_tensor, mask_tensor, fname

class SMPLightningModule(pl.LightningModule):
    def __init__(self, cfg):
        super().__init__()
        self.save_hyperparameters(ignore=['cfg'])
        self.cfg = cfg
        self.lr = cfg.LEARNING_RATE
        
        model_params = dict(
            encoder_name=cfg.ENCODER, 
            encoder_weights=cfg.ENCODER_WEIGHTS, 
            in_channels=3, 
            classes=cfg.NUM_CLASSES
        )
        
        if cfg.ACTIVE_MODEL == "unet":
            self.model = smp.Unet(**model_params)
        elif cfg.ACTIVE_MODEL == "deeplabv3":
            self.model = smp.DeepLabV3(**model_params)
        elif cfg.ACTIVE_MODEL == "segformer":
            self.model = smp.Segformer(**model_params)
        else:
            raise ValueError(f"Unsupported model: {cfg.ACTIVE_MODEL}")

        if cfg.LOSS_NAME == "bce":
            self.criterion = torch.nn.BCEWithLogitsLoss()
        elif cfg.LOSS_NAME == "dice":
            self.criterion = smp.losses.DiceLoss(mode=smp.losses.BINARY_MODE, from_logits=True)
        elif cfg.LOSS_NAME == "focal":
            self.criterion = smp.losses.FocalLoss(mode=smp.losses.BINARY_MODE)
        elif cfg.LOSS_NAME == "jaccard":
            self.criterion = smp.losses.JaccardLoss(mode=smp.losses.BINARY_MODE, from_logits=True)
        else:
            raise ValueError(f"Unsupported loss function: {cfg.LOSS_NAME}")
        
        self.conf_matrix_metric = BinaryConfusionMatrix()
        
        self.training_step_losses = []
        self.validation_step_losses = []
        self.step_image_ious = {}
        
        self.train_epoch_losses = []
        self.val_epoch_losses = []
        self.val_epoch_ious = []
        self.val_epoch_confusion_matrices = []
        self.val_epoch_image_ious = []

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        images, masks, _ = batch
        logits = self(images)
        loss = self.criterion(logits, masks)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.training_step_losses.append(loss.item())
        return loss

    def on_train_epoch_end(self):
        if self.training_step_losses:
            avg_loss = sum(self.training_step_losses) / len(self.training_step_losses)
            self.train_epoch_losses.append(avg_loss)
            self.training_step_losses.clear()

    def validation_step(self, batch, batch_idx):
        images, masks, fnames = batch
        logits = self(images)
        loss = self.criterion(logits, masks)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.validation_step_losses.append(loss.item())
        
        preds = (logits > 0).float() 
        self.conf_matrix_metric.update(preds.flatten(), masks.flatten().long())
        
        for i in range(len(fnames)):
            p = preds[i]
            m = masks[i]
            intersection = torch.logical_and(p == 1, m == 1).sum().item()
            union = torch.logical_or(p == 1, m == 1).sum().item()
            
            if union == 0:
                iou = 1.0 if torch.all(p == 0) and torch.all(m == 0) else 0.0
            else:
                iou = intersection / union
                
            self.step_image_ious[fnames[i]] = iou
            
        return loss

    def on_validation_epoch_end(self):
        if self.validation_step_losses:
            avg_loss = sum(self.validation_step_losses) / len(self.validation_step_losses)
            self.val_epoch_losses.append(avg_loss)
            self.validation_step_losses.clear()
        
        conf_mat = self.conf_matrix_metric.compute()
        tn, fp, fn, tp = conf_mat[0,0], conf_mat[0,1], conf_mat[1,0], conf_mat[1,1]
        
        union = tp + fp + fn
        iou = (tp / union).item() if union > 0 else 0.0
        
        self.val_epoch_ious.append(iou)
        self.val_epoch_confusion_matrices.append(conf_mat.cpu().numpy().tolist())
        self.log("val_iou", iou, prog_bar=True)
        
        self.val_epoch_image_ious.append(self.step_image_ious.copy())
        self.step_image_ious.clear()
        self.conf_matrix_metric.reset()

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

class SegmentationExperiment:
    def __init__(self, config):
        self.cfg = config
        self.all_filenames = []
        self.strat_labels = [] 
        self._prepare_dataset()
        self.preprocessing_fn = smp.encoders.get_preprocessing_fn(self.cfg.ENCODER, self.cfg.ENCODER_WEIGHTS)
        os.makedirs(self.cfg.RESULTS_DIR, exist_ok=True)

    def _prepare_dataset(self):
        raw_files = sorted([f for f in os.listdir(self.cfg.IMAGE_DIR) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
        for fname in raw_files:
            category = fname.split('-')[0].lower()
            if category == "clean" and self.cfg.INCLUDE_CLEAN_DATA:
                self.all_filenames.append(fname)
                self.strat_labels.append("clean")
            elif category in self.cfg.INCLUDED_FILTH_TYPES:
                self.all_filenames.append(fname)
                self.strat_labels.append(category)

    def run_k_fold(self):
        skf = StratifiedKFold(n_splits=self.cfg.K_FOLDS, shuffle=True, random_state=42)
        X = np.array(self.all_filenames)
        y = np.array(self.strat_labels)
        
        experiment_results = {"config": vars(self.cfg), "folds": {}}

        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
            fold_num = fold + 1
            print(f"\n{'='*60}\n🔥 FOLD {fold_num}/{self.cfg.K_FOLDS} | {self.cfg.get_experiment_name()}\n{'='*60}")
            
            train_dataset = UnifiedSegmentationDataset(X[train_idx].tolist(), self.cfg.IMAGE_DIR, self.cfg.MASK_DIR, self.preprocessing_fn)
            val_dataset = UnifiedSegmentationDataset(X[val_idx].tolist(), self.cfg.IMAGE_DIR, self.cfg.MASK_DIR, self.preprocessing_fn)
            
            train_loader = DataLoader(train_dataset, batch_size=self.cfg.TRAIN_BATCH_SIZE, shuffle=True, num_workers=0)
            val_loader = DataLoader(val_dataset, batch_size=self.cfg.EVAL_BATCH_SIZE, shuffle=False, num_workers=0)

            model = SMPLightningModule(self.cfg)

            checkpoint_callback = ModelCheckpoint(
                dirpath=f"./final_models/{self.cfg.get_experiment_name()}_fold_{fold_num}",
                filename="best-model",
                save_top_k=1,
                monitor="val_loss",
                mode="min"
            )
            early_stop_callback = EarlyStopping(monitor="val_loss", patience=self.cfg.PATIENCE, mode="min")

            trainer = pl.Trainer(
                max_epochs=self.cfg.EPOCHS,
                accelerator="gpu" if torch.cuda.is_available() else "cpu",
                devices=1,
                callbacks=[early_stop_callback, checkpoint_callback],
                enable_progress_bar=True,
                logger=False,
                num_sanity_val_steps=0 
            )

            trainer.fit(model, train_loader, val_loader)
            
            experiment_results["folds"][f"fold_{fold_num}"] = {
                "train_loss": model.train_epoch_losses,
                "val_loss": model.val_epoch_losses,
                "val_iou": model.val_epoch_ious,
                "val_confusion_matrices": model.val_epoch_confusion_matrices,
                "val_image_ious": model.val_epoch_image_ious # Save our new per-image data
            }

        result_path = os.path.join(self.cfg.RESULTS_DIR, f"{self.cfg.get_experiment_name()}_metrics.json")
        with open(result_path, 'w') as f:
            json.dump(experiment_results, f, indent=4)
            
        print(f"\nTraining complete! Metrics saved to: {result_path}")

class ExperimentReporter:
    def __init__(self, results_dir="./results", graphs_dir="./graphs"):
        self.results_dir = results_dir
        self.graphs_dir = graphs_dir
        os.makedirs(self.graphs_dir, exist_ok=True)

    def load_experiment_data(self, cfg):
        experiment_name = cfg.get_experiment_name()
        file_path = os.path.join(self.results_dir, f"{experiment_name}_metrics.json")
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"No results found for {experiment_name} at {file_path}")
        with open(file_path, 'r') as f:
            return json.load(f)

    def plot_kfold_history(self, cfg):
        data = self.load_experiment_data(cfg)
        experiment_name = cfg.get_experiment_name()
        folds = data["folds"]
        num_folds = len(folds)
        
        fig, axes = plt.subplots(2, num_folds, figsize=(6 * num_folds, 8))
        if num_folds == 1: axes = np.expand_dims(axes, axis=1)

        for i, (fold_name, metrics) in enumerate(folds.items()):
            epochs = range(1, len(metrics["train_loss"]) + 1)
            
            ax_loss = axes[0, i]
            ax_loss.plot(epochs, metrics["train_loss"], label='Train Loss', color='blue')
            ax_loss.plot(epochs, metrics["val_loss"], label='Val Loss', color='orange')
            ax_loss.set_title(f"{fold_name.replace('_', ' ').title()} - Loss")
            ax_loss.set_xlabel("Epochs")
            ax_loss.set_ylabel("Loss")
            ax_loss.legend()
            ax_loss.grid(True, linestyle='--', alpha=0.7)

            ax_iou = axes[1, i]
            ax_iou.plot(epochs, metrics["val_iou"], label='Val IoU', color='green')
            ax_iou.set_title(f"{fold_name.replace('_', ' ').title()} - Validation IoU")
            ax_iou.set_xlabel("Epochs")
            ax_iou.set_ylabel("IoU Score")
            ax_iou.legend()
            ax_iou.grid(True, linestyle='--', alpha=0.7)

        plt.suptitle(f"{experiment_name.upper()} - K-Fold History", fontsize=16, y=1.02)
        plt.tight_layout()
        save_path = os.path.join(self.graphs_dir, f"{experiment_name}_kfold_history.png")
        plt.savefig(save_path, bbox_inches='tight')
        print(f"📊 K-Fold graph saved to: {save_path}")

    def plot_confusion_matrix(self, cfg, fold_num=1):
        data = self.load_experiment_data(cfg)
        experiment_name = cfg.get_experiment_name()
        fold_key = f"fold_{fold_num}"
        if fold_key not in data["folds"]: return

        best_epoch_idx = np.argmax(data["folds"][fold_key]["val_iou"])
        conf_mat = np.array(data["folds"][fold_key]["val_confusion_matrices"][best_epoch_idx])
        conf_mat_pct = (conf_mat / np.sum(conf_mat)) * 100

        plt.figure(figsize=(6, 5))
        labels = ["Background (0)", "Filth (1)"]
        
        sns.heatmap(conf_mat_pct, annot=True, fmt=".2f", cmap="Greens", xticklabels=labels, yticklabels=labels)
        for t in plt.gca().texts: t.set_text(t.get_text() + " %")
        
        plt.title(f"{experiment_name.upper()} - {fold_key.title()} Normalized CM\n(Best Epoch: {best_epoch_idx + 1})")
        plt.ylabel("True Label")
        plt.xlabel("Predicted Label")

        save_path = os.path.join(self.graphs_dir, f"{experiment_name}_{fold_key}_confusion_matrix.png")
        plt.savefig(save_path, bbox_inches='tight')
        plt.close()

    def plot_averaged_confusion_matrix(self, cfg):
        data = self.load_experiment_data(cfg)
        experiment_name = cfg.get_experiment_name()
        folds = data["folds"]
        if not folds: return

        aggregated_conf_mat = np.zeros((2, 2), dtype=np.int64)
        for fold_key, metrics in folds.items():
            best_epoch_idx = np.argmax(metrics["val_iou"])
            aggregated_conf_mat += np.array(metrics["val_confusion_matrices"][best_epoch_idx])

        conf_mat_pct = (aggregated_conf_mat / np.sum(aggregated_conf_mat)) * 100

        plt.figure(figsize=(6, 5))
        labels = ["Background (0)", "Filth (1)"]
        
        sns.heatmap(conf_mat_pct, annot=True, fmt=".2f", cmap="Greens", xticklabels=labels, yticklabels=labels)
        for t in plt.gca().texts: t.set_text(t.get_text() + " %")
        
        plt.title(f"{experiment_name.upper()} - Averaged Normalized CM (All Folds)")
        plt.ylabel("True Label")
        plt.xlabel("Predicted Label")

        save_path = os.path.join(self.graphs_dir, f"{experiment_name}_averaged_confusion_matrix.png")
        plt.savefig(save_path, bbox_inches='tight')
        plt.close()


In [ ]:
class ModelBenchmarker:
    def __init__(self, cfg, fold_num=1, input_shape=(1, 3, 512, 512), num_runs=100):
        self.cfg = cfg
        self.fold_num = fold_num
        self.input_shape = input_shape
        self.num_runs = num_runs
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        self.checkpoint_path = os.path.join(
            "./final_models", 
            f"{self.cfg.get_experiment_name()}_fold_{self.fold_num}", 
            "best-model.ckpt"
        )
        
        if not os.path.exists(self.checkpoint_path):
            raise FileNotFoundError(
                f"Checkpoint not found at: {self.checkpoint_path}\n"
                f"Make sure you have trained {self.cfg.get_experiment_name()} for fold {self.fold_num} before benchmarking."
            )

    def run(self):
        print(f"\n⏳ Benchmarking {self.cfg.ACTIVE_MODEL.upper()} ({self.cfg.ENCODER}) on {str(self.device).upper()}...")
        
        model = SMPLightningModule.load_from_checkpoint(self.checkpoint_path, cfg=self.cfg)
        model.to(self.device)
        model.eval()
        
        dummy_input = torch.randn(self.input_shape, device=self.device)
        
        if self.device.type == "cuda":
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(device=self.device)
            
        with torch.no_grad():
            _ = model(dummy_input) 
            
        if self.device.type == "cuda":
            peak_vram_mb = torch.cuda.max_memory_allocated(device=self.device) / (1024 * 1024)
        else:
            peak_vram_mb = 0.0 
            
        print("Warming up...")
        with torch.no_grad():
            for _ in range(10):
                _ = model(dummy_input)
                
        print(f"Running {self.num_runs} timed iterations...")
        
        if self.device.type == "cuda":
            starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
            timings = np.zeros((self.num_runs, 1))
            
            with torch.no_grad():
                for i in range(self.num_runs):
                    starter.record()
                    _ = model(dummy_input)
                    ender.record()
                    
                    torch.cuda.synchronize()
                    timings[i] = starter.elapsed_time(ender)
                    
        else:
            timings = np.zeros((self.num_runs, 1))
            with torch.no_grad():
                for i in range(self.num_runs):
                    start_time = time.perf_counter()
                    _ = model(dummy_input)
                    end_time = time.perf_counter()
                    timings[i] = (end_time - start_time) * 1000 

        mean_syn_ms = np.mean(timings)
        std_syn_ms = np.std(timings)
        fps = 1000.0 / mean_syn_ms
        
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

        print("\n" + "="*50)
        print(f"BENCHMARK RESULTS: {self.cfg.ACTIVE_MODEL.upper()} ({self.cfg.ENCODER})")
        print("="*50)
        print(f"Input Shape      : {self.input_shape}")
        print(f"Total Params     : {total_params:,}")
        print(f"Trainable Params : {trainable_params:,}")
        print("-" * 50)
        print(f"Peak VRAM Usage  : {peak_vram_mb:.2f} MB")
        print(f"Latency per img  : {mean_syn_ms:.2f} ms (± {std_syn_ms:.2f} ms)")
        print(f"Throughput       : {fps:.2f} FPS")
        print("="*50 + "\n")
        
        return {
            "params": total_params,
            "vram_mb": peak_vram_mb,
            "latency_ms": mean_syn_ms,
            "fps": fps
        }

In [ ]:
reporter = ExperimentReporter()

In [ ]:
cfg_unet = Config(active_model="unet", encoder_name="resnet34", k_folds=5, train_batch_size=8)
SegmentationExperiment(cfg_unet).run_k_fold()

reporter.plot_kfold_history(cfg_unet)
reporter.plot_averaged_confusion_matrix(cfg_unet)
ModelBenchmarker(cfg=cfg_unet, fold_num=1).run()

In [ ]:
cfg_segformer = Config(active_model="segformer", encoder_name="mit_b0", k_folds=2, train_batch_size=8)


In [ ]:
SegmentationExperiment(cfg_segformer).run_k_fold()

reporter.plot_kfold_history(cfg_segformer)
reporter.plot_averaged_confusion_matrix(cfg_segformer)
ModelBenchmarker(cfg=cfg_segformer, fold_num=1).run()

In [ ]:
cfg_deeplab = Config(active_model="deeplabv3", encoder_name="resnet34", k_folds=2, train_batch_size=8)
SegmentationExperiment(cfg_deeplab).run_k_fold()

reporter.plot_kfold_history(cfg_deeplab)
reporter.plot_averaged_confusion_matrix(cfg_deeplab)
ModelBenchmarker(cfg=cfg_deeplab, fold_num=1).run()

In [ ]:
config = cfg_segformer